# 05. Random Forest - Breast Cancer Classification**Topic:** Compare a single decision tree vs a random forest. Demonstrate how ensembling reduces variance.**Dataset:** Breast Cancer Wisconsin (sklearn).

In [ ]:
# Run this cell first if running locally. In Google Colab everything is pre-installed.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)np.random.seed(42)

In [ ]:
from sklearn.datasets import load_breast_cancerfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrixdata = load_breast_cancer(as_frame=True)X, y = data.data, data.targetprint(X.shape)print(data.target_names)

## 1. Single tree vs Random Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)tree   = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)forest = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)for name, m in [("Decision Tree", tree), ("Random Forest", forest)]:    pred = m.predict(X_test)    proba = m.predict_proba(X_test)[:, 1]    print(f"{name:15s}  Acc = {accuracy_score(y_test, pred):.3f}   AUC = {roc_auc_score(y_test, proba):.3f}")

## 2. Effect of n_estimators

In [ ]:
ns = [1, 5, 10, 25, 50, 100, 200, 500]scores = []for n in ns:    rf = RandomForestClassifier(n_estimators=n, random_state=42)    s = cross_val_score(rf, X_train, y_train, cv=5, scoring="accuracy").mean()    scores.append(s)plt.figure(figsize=(10, 5))plt.plot(ns, scores, "o-", color="black")plt.xscale("log")plt.xlabel("n_estimators")plt.ylabel("CV accuracy")plt.title("More trees = more stable predictions")plt.show()

## 3. Feature importance

In [ ]:
imp = pd.Series(forest.feature_importances_, index=X.columns).sort_values().tail(15)plt.figure(figsize=(8, 6))imp.plot.barh(color="gray")plt.title("Top 15 most important features")plt.show()

## 4. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, forest.predict(X_test))sns.heatmap(cm, annot=True, fmt="d", cmap="Greys",            xticklabels=data.target_names, yticklabels=data.target_names)plt.xlabel("Predicted")plt.ylabel("Actual")plt.show()

## 5. Exercises1. **Try `max_features="sqrt"` vs `"log2"` vs `1.0`** and compare.2. **Try `RandomForestRegressor`** on the California Housing dataset.3. **Use OOB score** (`oob_score=True`, `bootstrap=True`) instead of CV.4. **Time the training** of 1000 trees vs 100. How does it scale?